# Module 6: Text-to-SQL Agent — End-to-End Training Pipeline

## Learning Objectives

By completing this notebook, you will:
1. Build a SQLite company database with realistic schema and data
2. Implement a FastMCP server exposing three database tools to an agent
3. Auto-generate training scenarios of varying difficulty from tool schemas
4. Score agent responses with RULER before using it in training
5. Implement the rollout function connecting question, tool calls, and answer
6. Run a complete GRPO training loop and interpret training logs

## Prerequisites
- Modules 00-05 complete (GRPO, ART framework, RULER rewards, MCP integration, training loops)
- SQLite familiarity: CREATE TABLE, SELECT, JOIN
- Python 3.10+

## Estimated Time: 15 minutes

---

> **What you are building:** A Qwen2.5-3B model that starts knowing nothing about your database schema and learns — through self-generated rollouts scored by o4-mini — to explore the schema with tools and write correct multi-table SQL queries.

## 1. Setup and Configuration

All dependencies and configuration values live here. Change `MODEL`, `GROUP_SIZE`, or `NUM_STEPS` in this cell — the rest of the notebook picks them up automatically.

- **`MODEL`**: The policy model ART loads into vLLM and fine-tunes with Unsloth LoRA.
- **`JUDGE_MODEL`**: The external judge RULER calls to score each rollout. Must be a frontier model — it needs to reliably evaluate SQL answer correctness.
- **`GROUP_SIZE`**: How many rollouts to generate per scenario per step. GRPO needs multiple samples to compute relative advantages — 4 is the minimum meaningful value.
- **`NUM_STEPS`**: Training steps. 50 steps is enough to see measurable improvement on this schema; 200+ for convergence.

In [ ]:
# Purpose: Install dependencies and set configuration for the full pipeline.
# Run this cell once. art-trainer pulls in vLLM and Unsloth automatically.

# !pip install art-trainer fastmcp vllm

import asyncio
import json
import os
import sqlite3
import subprocess
import sys
import textwrap
import time
from pathlib import Path
from typing import Any

# ---------------------------------------------------------------------------
# Training configuration — modify these to experiment
# ---------------------------------------------------------------------------
MODEL       = "Qwen/Qwen2.5-3B"   # Policy model loaded into vLLM
JUDGE_MODEL = "openai/o4-mini"     # LLM-as-judge for RULER scoring
GROUP_SIZE  = 4                    # Rollouts per scenario per step (GRPO minimum: 4)
NUM_STEPS   = 50                   # Training steps (200+ for full convergence)

DB_PATH     = Path("company.db")   # SQLite database written by Section 2
SERVER_PORT = 8000                 # MCP server port
MCP_URL     = f"http://localhost:{SERVER_PORT}"

print(f"Policy model : {MODEL}")
print(f"Judge model  : {JUDGE_MODEL}")
print(f"Group size   : {GROUP_SIZE}")
print(f"Training steps: {NUM_STEPS}")
print(f"Database path: {DB_PATH}")
print(f"MCP URL      : {MCP_URL}")

## 2. Create the Company Database

The database determines what the agent can learn. A schema with real cardinality, real foreign-key relationships, and real variation forces the agent to develop general reasoning: how to explore an unknown schema, when to JOIN, how to filter correctly.

**Schema overview:**

```
departments  (id, name, location, budget, headcount)
    │
    ├── employees (id, name, department_id FK, salary, role, hire_date, status)
    │
    └── projects  (id, name, department_id FK, budget, status, start_date, end_date)
```

Both `employees` and `projects` reference `departments` — not each other. This forces the agent to navigate indirect relationships, which is the realistic case in production databases.

The `status` column on employees (`active` / `on_leave` / `terminated`) is deliberate: many interesting queries require filtering by status, and the agent must discover this column exists before using it.

In [ ]:
# Purpose: Create and populate the company SQLite database used for agent training.
# Key Concept: Real, irregular data forces the agent to write general queries
#              rather than pattern-matching on suspiciously round values.


def create_company_database(db_path: Path = DB_PATH) -> sqlite3.Connection:
    """
    Create and populate the company database.

    Parameters
    ----------
    db_path : Path
        Filesystem path for the SQLite database file.
        Existing file is deleted and recreated from scratch.

    Returns
    -------
    sqlite3.Connection
        Open connection to the created database.
    """
    # Start clean — delete any prior run's database
    if db_path.exists():
        db_path.unlink()

    conn = sqlite3.connect(str(db_path))
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    # SQLite disables foreign-key enforcement by default.
    # Enabling it ensures orphan rows get rejected at insert time.
    cursor.execute("PRAGMA foreign_keys = ON")

    # ------------------------------------------------------------------
    # Table 1: departments — the anchor table both other tables reference
    # ------------------------------------------------------------------
    cursor.execute("""
        CREATE TABLE departments (
            id        INTEGER PRIMARY KEY AUTOINCREMENT,
            name      TEXT    NOT NULL UNIQUE,
            location  TEXT    NOT NULL,
            budget    REAL    NOT NULL,
            headcount INTEGER NOT NULL
        )
    """)

    # Irregular budget values catch agents that hardcode answers
    departments = [
        ("Engineering",  "San Francisco", 4_200_000.00, 42),
        ("Product",      "San Francisco", 1_800_000.00, 18),
        ("Sales",        "New York",      2_100_000.00, 31),
        ("Marketing",    "New York",        950_000.00, 12),
        ("Finance",      "Chicago",       1_100_000.00,  9),
        ("Legal",        "Chicago",         620_000.00,  5),
        ("Data Science", "San Francisco", 2_800_000.00, 22),
        ("Operations",   "Austin",        1_350_000.00, 19),
    ]
    cursor.executemany(
        "INSERT INTO departments (name, location, budget, headcount) VALUES (?, ?, ?, ?)",
        departments,
    )

    # ------------------------------------------------------------------
    # Table 2: employees — references departments via department_id FK
    # ------------------------------------------------------------------
    cursor.execute("""
        CREATE TABLE employees (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            name          TEXT    NOT NULL,
            department_id INTEGER NOT NULL REFERENCES departments(id),
            salary        REAL    NOT NULL,
            role          TEXT    NOT NULL,
            hire_date     TEXT    NOT NULL,
            status        TEXT    NOT NULL DEFAULT 'active'
                          CHECK (status IN ('active', 'on_leave', 'terminated'))
        )
    """)

    # dept_id: Engineering=1, Product=2, Sales=3, Marketing=4,
    #          Finance=5, Legal=6, Data Science=7, Operations=8
    employees = [
        # Engineering
        ("Alice Chen",       1, 165_000, "Principal Engineer",    "2018-03-12", "active"),
        ("Bob Martinez",     1, 142_000, "Senior Engineer",       "2019-07-22", "active"),
        ("Carol Patel",      1, 138_000, "Senior Engineer",       "2020-01-08", "active"),
        ("David Kim",        1, 118_000, "Engineer",              "2021-05-14", "active"),
        ("Elena Novak",      1, 122_000, "Engineer",              "2021-08-30", "active"),
        ("Frank Liu",        1, 195_000, "Engineering Manager",   "2016-09-01", "active"),
        ("Grace Thompson",   1, 115_000, "Engineer",              "2022-02-14", "active"),
        ("Henry Park",       1, 125_000, "Senior Engineer",       "2020-11-03", "on_leave"),
        # Product
        ("Iris Johnson",     2, 148_000, "Product Manager",       "2019-04-15", "active"),
        ("James Wilson",     2, 135_000, "Product Manager",       "2020-06-20", "active"),
        ("Kate Brown",       2, 158_000, "Director of Product",   "2017-10-10", "active"),
        ("Liam Davis",       2, 120_000, "Associate PM",          "2022-09-01", "active"),
        # Sales
        ("Maya Okonkwo",     3,  95_000, "Account Executive",     "2020-03-01", "active"),
        ("Noah Garcia",      3,  88_000, "Account Executive",     "2021-01-15", "active"),
        ("Olivia Smith",     3, 112_000, "Sales Manager",         "2018-11-20", "active"),
        ("Peter Zhang",      3,  92_000, "Account Executive",     "2021-07-08", "active"),
        ("Quinn Taylor",     3,  78_000, "SDR",                   "2023-01-10", "active"),
        ("Rachel Moore",     3,  82_000, "SDR",                   "2022-08-22", "terminated"),
        # Marketing
        ("Sam Lee",          4, 105_000, "Marketing Manager",     "2019-06-15", "active"),
        ("Tina Hernandez",   4,  88_000, "Content Strategist",    "2021-03-01", "active"),
        ("Uma Patel",        4,  92_000, "Growth Marketer",       "2020-09-14", "active"),
        ("Victor Russo",     4, 115_000, "Director of Marketing", "2017-02-28", "active"),
        # Finance
        ("Wendy Chen",       5, 132_000, "Financial Analyst",     "2019-08-19", "active"),
        ("Xavier Rodriguez", 5, 145_000, "Finance Manager",       "2018-04-03", "active"),
        ("Yara Johansson",   5, 128_000, "Financial Analyst",     "2020-12-07", "active"),
        # Legal
        ("Zoe Williams",     6, 185_000, "Senior Counsel",        "2018-07-16", "active"),
        ("Aaron Brooks",     6, 165_000, "Counsel",               "2020-05-18", "active"),
        # Data Science
        ("Bella Nakamura",   7, 158_000, "Senior Data Scientist",    "2019-02-11", "active"),
        ("Carlos Mendez",    7, 145_000, "Data Scientist",           "2020-10-05", "active"),
        ("Diana Walsh",      7, 172_000, "Principal Data Scientist", "2017-08-20", "active"),
        ("Ethan Foster",     7, 138_000, "Data Scientist",           "2021-04-19", "active"),
        ("Fiona Chang",      7, 180_000, "Data Science Manager",     "2016-11-30", "active"),
        ("Gus Andersen",     7, 132_000, "Data Scientist",           "2022-01-10", "on_leave"),
        # Operations
        ("Hannah Scott",     8,  98_000, "Operations Analyst",   "2020-07-06", "active"),
        ("Ian Powell",       8, 115_000, "Operations Manager",   "2018-09-12", "active"),
        ("Julia Moreau",     8,  92_000, "Operations Analyst",   "2021-11-30", "active"),
        ("Kevin Singh",      8,  88_000, "Coordinator",          "2022-06-15", "active"),
    ]
    cursor.executemany(
        """INSERT INTO employees (name, department_id, salary, role, hire_date, status)
           VALUES (?, ?, ?, ?, ?, ?)""",
        employees,
    )

    # ------------------------------------------------------------------
    # Table 3: projects — also references departments, not employees
    # This models the realistic case where a project belongs to a
    # department, not a single person.
    # ------------------------------------------------------------------
    cursor.execute("""
        CREATE TABLE projects (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            name          TEXT    NOT NULL,
            department_id INTEGER NOT NULL REFERENCES departments(id),
            budget        REAL    NOT NULL,
            status        TEXT    NOT NULL
                          CHECK (status IN ('planning', 'active', 'completed', 'cancelled')),
            start_date    TEXT    NOT NULL,
            end_date      TEXT
        )
    """)

    projects = [
        # Engineering
        ("Project Falcon",      1,  850_000, "active",    "2024-01-15", None),
        ("API Gateway Rebuild", 1,  420_000, "completed", "2023-06-01", "2024-02-28"),
        ("Mobile SDK v3",       1,  310_000, "active",    "2024-03-01", None),
        ("Security Hardening",  1,  275_000, "planning",  "2024-07-01", None),
        # Product
        ("UX Redesign 2024",    2,  380_000, "active",    "2024-02-01", None),
        ("Onboarding Flow",     2,  145_000, "completed", "2023-09-01", "2024-01-31"),
        # Sales
        ("Enterprise Pipeline", 3,  220_000, "active",    "2024-01-01", None),
        ("APAC Expansion",      3,  540_000, "planning",  "2024-08-01", None),
        # Marketing
        ("Brand Refresh",       4,  280_000, "active",    "2024-01-20", None),
        ("Q3 Campaign",         4,  165_000, "completed", "2023-07-01", "2023-09-30"),
        # Finance
        ("ERP Migration",       5,  620_000, "active",    "2023-11-01", None),
        ("Audit Automation",    5,  180_000, "planning",  "2024-06-01", None),
        # Data Science
        ("Churn Prediction v2", 7,  490_000, "active",    "2024-02-15", None),
        ("Demand Forecasting",  7,  375_000, "completed", "2023-04-01", "2023-12-31"),
        ("Real-time Anomaly",   7,  280_000, "active",    "2024-04-01", None),
        ("LLM Integration",     7,  720_000, "planning",  "2024-09-01", None),
        # Operations
        ("Warehouse Automation",8,  410_000, "active",    "2024-03-01", None),
        ("Vendor Consolidation",8,   95_000, "completed", "2023-08-01", "2024-01-15"),
    ]
    cursor.executemany(
        """INSERT INTO projects (name, department_id, budget, status, start_date, end_date)
           VALUES (?, ?, ?, ?, ?, ?)""",
        projects,
    )

    conn.commit()
    return conn


def verify_database(conn: sqlite3.Connection) -> None:
    """
    Run diagnostic queries to confirm the database is correct.

    Parameters
    ----------
    conn : sqlite3.Connection
        Open connection to the database to verify.
    """
    cursor = conn.cursor()

    # Step 1: Row counts — catches any truncated inserts immediately
    print("Row counts:")
    for table in ("departments", "employees", "projects"):
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        print(f"  {table}: {cursor.fetchone()[0]} rows")

    # Step 2: Referential integrity — any orphan employees?
    cursor.execute("""
        SELECT COUNT(*) FROM employees e
        LEFT JOIN departments d ON e.department_id = d.id
        WHERE d.id IS NULL
    """)
    print(f"  Orphan employees (expect 0): {cursor.fetchone()[0]}")

    # Step 3: Sample JOIN — confirms the cross-table queries the agent will use work
    cursor.execute("""
        SELECT d.name, COUNT(e.id) AS headcount, ROUND(AVG(e.salary), 0) AS avg_salary
        FROM departments d
        JOIN employees e ON d.id = e.department_id AND e.status = 'active'
        GROUP BY d.name
        ORDER BY avg_salary DESC
        LIMIT 3
    """)
    print("\nTop 3 departments by average active salary:")
    for row in cursor.fetchall():
        print(f"  {row[0]}: {row[1]} employees, avg ${row[2]:,.0f}")


# Create the database and verify it
conn = create_company_database(DB_PATH)
verify_database(conn)
conn.close()
print(f"\nDatabase written to {DB_PATH}")

## 3. Build the MCP Server

The agent never reads the database file directly. Everything flows through three MCP tools:

| Tool | Arguments | Purpose |
|------|-----------|--------|
| `list_tables()` | none | Discover what tables exist |
| `describe_table(table_name)` | table name | Inspect columns and foreign keys |
| `run_query(sql)` | SELECT statement | Execute a query and get results |

This constraint is intentional. The agent must discover the schema through tools, the same way a human database analyst would approach an unfamiliar database. What the agent learns here transfers directly to any production database it encounters.

Notice `run_query` returns errors as structured data rather than raising exceptions. The agent reads the error and can correct its SQL — recovering from `"no such column"` is one of the most valuable behaviors training teaches.

After defining the server module, we start it as a background subprocess so the rest of the notebook can connect to it.

In [ ]:
# Purpose: Write the FastMCP server to a file and launch it as a background process.
# Key Concept: The server runs persistently — it handles tool calls during every rollout
#              without restarting between them.

SERVER_SCRIPT = Path("company_db_server.py")

SERVER_SOURCE = '''
"""
company_db_server.py

FastMCP server exposing the company database via three read-only tools.
Start this before running the training notebook.
"""

import sqlite3
from typing import Any
from fastmcp import FastMCP

DB_PATH = "company.db"

mcp = FastMCP(
    name="company-database",
    description=(
        "Tools for querying the company database. "
        "Always call list_tables first, then describe_table before writing SQL."
    ),
)


def _connect() -> sqlite3.Connection:
    """Open a fresh per-call connection to avoid SQLite threading issues."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


@mcp.tool()
def list_tables() -> list[str]:
    """
    Return the names of all tables in the company database.
    Always call this first before writing any SQL.
    """
    conn = _connect()
    try:
        cur = conn.cursor()
        cur.execute("SELECT name FROM sqlite_master WHERE type=\'table\' ORDER BY name")
        return [row["name"] for row in cur.fetchall()]
    finally:
        conn.close()


@mcp.tool()
def describe_table(table_name: str) -> dict[str, Any]:
    """
    Return columns, types, and foreign keys for table_name.
    Call this before writing a JOIN or filtering on a column.

    Args:
        table_name: Exact table name from list_tables(). Case-sensitive.
    """
    conn = _connect()
    try:
        cur = conn.cursor()
        cur.execute(
            "SELECT name FROM sqlite_master WHERE type=\'table\' AND name=?",
            (table_name,),
        )
        if cur.fetchone() is None:
            return {"error": f"Table \'{table_name}\' not found. Call list_tables()."}

        cur.execute(f"PRAGMA table_info({table_name})")
        columns = [
            {
                "name": r["name"],
                "type": r["type"],
                "nullable": not bool(r["notnull"]),
                "primary_key": bool(r["pk"]),
            }
            for r in cur.fetchall()
        ]

        cur.execute(f"PRAGMA foreign_key_list({table_name})")
        foreign_keys = [
            {"column": r["from"], "references": f"{r[\'table\']}({r[\'to\']})"}  
            for r in cur.fetchall()
        ]

        return {"table": table_name, "columns": columns, "foreign_keys": foreign_keys}
    finally:
        conn.close()


@mcp.tool()
def run_query(sql: str) -> dict[str, Any]:
    """
    Execute a SELECT query and return results as a list of row dicts.
    Only SELECT is permitted — write operations are rejected.

    Args:
        sql: A valid SQLite SELECT statement.
    """
    # Reject non-SELECT statements before touching the database
    if not sql.strip().upper().startswith("SELECT"):
        return {
            "error": "Only SELECT statements are permitted.",
            "sql_attempted": sql[:80],
        }

    conn = _connect()
    try:
        cur = conn.cursor()
        cur.execute(sql)
        rows = [dict(r) for r in cur.fetchall()]
        return {"rows": rows, "row_count": len(rows)}
    except sqlite3.OperationalError as exc:
        # Structured error: the agent reads this and corrects its SQL
        return {"error": str(exc), "sql_attempted": sql[:200]}
    finally:
        conn.close()


if __name__ == "__main__":
    print("Starting company-database MCP server on port 8000...")
    mcp.run(transport="streamable-http", host="localhost", port=8000)
'''

# Write the server script to disk
SERVER_SCRIPT.write_text(SERVER_SOURCE)
print(f"Server script written to {SERVER_SCRIPT}")

# Launch the server as a background subprocess.
# We redirect stdout/stderr to PIPE so the notebook doesn't block on server output.
server_proc = subprocess.Popen(
    [sys.executable, str(SERVER_SCRIPT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Give the server a moment to bind its port before we verify the connection
time.sleep(2)

if server_proc.poll() is None:
    print(f"MCP server started (PID {server_proc.pid})")
    print(f"Listening on {MCP_URL}")
else:
    stderr_output = server_proc.stderr.read().decode()
    raise RuntimeError(f"Server failed to start:\n{stderr_output}")

In [ ]:
# Purpose: Verify the ART client can discover all three tools before training starts.
# Key Concept: If any tool is missing here, training will fail silently during rollouts.

import art


async def verify_mcp_connection(url: str) -> list[str]:
    """
    Connect to the MCP server and return discovered tool names.

    Parameters
    ----------
    url : str
        Base URL of the running MCP server.

    Returns
    -------
    list[str]
        Names of all tools the server exposes.
    """
    client = art.MCPClient(url)
    async with client:
        tools = await client.list_tools()
    return [t.name for t in tools]


discovered = asyncio.run(verify_mcp_connection(MCP_URL))
print(f"Discovered tools: {discovered}")

REQUIRED_TOOLS = {"list_tables", "describe_table", "run_query"}
missing = REQUIRED_TOOLS - set(discovered)
if missing:
    raise RuntimeError(
        f"Missing required tools: {missing}. "
        "Is company_db_server.py running on port 8000?"
    )

print("All required tools available. Ready to train.")

## 4. Auto-Generate Training Scenarios

ART generates training scenarios by inspecting the tool schemas from the MCP server. You do not write scenarios by hand — ART infers what kinds of questions are answerable given the available tools and generates a diverse set automatically.

Scenarios are assigned difficulty levels based on how many tool calls are required to answer them and how many tables the query must span:

| Difficulty | Pattern | Example |
|------------|---------|--------|
| Simple | Single table, no JOIN | "List all tables" |
| Medium | Single JOIN, one aggregation | "Average salary by department" |
| Complex | Multi-table JOIN + filter + aggregation | "Who leads the most expensive active project and what department are they in?" |

ART ensures the generated scenario mix includes all three difficulty bands. The agent must learn simple cases before it can consistently handle complex ones — and GRPO's advantage computation naturally rewards progress at each level.

In [ ]:
# Purpose: Generate training scenarios from the MCP tool schemas.
# Key Concept: ART inspects tool metadata (descriptions, argument types) to produce
#              questions that are answerable with the available tools.
#              No manual labeling required.


async def generate_scenarios(url: str, n_scenarios: int = 30) -> list[dict]:
    """
    Auto-generate training scenarios from the running MCP server's tool schemas.

    Parameters
    ----------
    url : str
        MCP server URL.
    n_scenarios : int
        Total number of scenarios to generate across all difficulty levels.

    Returns
    -------
    list[dict]
        Each dict has 'question', 'difficulty', and 'ground_truth_sql' keys.
    """
    # Step 1: Let ART inspect the tool schemas
    client = art.MCPClient(url)
    async with client:
        schemas = await client.list_tools()

    # Step 2: ART generates scenarios from the schemas
    # The ScenarioGenerator infers question types from tool names and descriptions.
    # It knows 'describe_table' implies schema exploration questions,
    # 'run_query' implies data retrieval questions, and so on.
    generator = art.ScenarioGenerator(tool_schemas=schemas)
    scenarios = await generator.generate(
        count=n_scenarios,
        difficulty_mix={"simple": 0.3, "medium": 0.4, "complex": 0.3},
    )

    return scenarios


scenarios = asyncio.run(generate_scenarios(MCP_URL, n_scenarios=30))

# Inspect one scenario from each difficulty band
for difficulty in ("simple", "medium", "complex"):
    sample = next(s for s in scenarios if s["difficulty"] == difficulty)
    print(f"\n--- {difficulty.upper()} ---")
    print(f"Question      : {sample['question']}")
    print(f"Ground truth  : {sample['ground_truth_sql']}")

print(f"\nTotal scenarios generated: {len(scenarios)}")

# Expected output:
# --- SIMPLE ---
# Question      : List all tables in the database.
# Ground truth  : (calls list_tables)
#
# --- MEDIUM ---
# Question      : What is the average salary for each department?
# Ground truth  : SELECT d.name, AVG(e.salary) FROM departments d
#                 JOIN employees e ON d.id = e.department_id GROUP BY d.name
#
# --- COMPLEX ---
# Question      : Who leads the most expensive active project and
#                 what department are they in?
# Ground truth  : SELECT e.name, d.name FROM employees e
#                 JOIN departments d ON e.department_id = d.id
#                 JOIN projects p ON p.department_id = d.id
#                 WHERE p.status = 'active'
#                 ORDER BY p.budget DESC LIMIT 1

## 5. RULER Scoring Demo

RULER (Relative Universal LLM Evaluation with Rankings) uses an LLM judge to compare agent responses to a reference answer. It returns a scalar score between 0.0 and 1.0 that becomes the reward signal in GRPO training.

The key property: RULER evaluates correctness, not format. An agent that returns the right number expressed as `"There are 36 employees"` scores the same as one that returns `"36"`. An agent that adds the wrong filter and returns `"29"` scores near zero.

Before using RULER in training, we verify it gives the rankings we expect on a controlled example. If the judge consistently misranks responses on simple cases, stop here and debug before running 50 training steps.

In [ ]:
# Purpose: Verify RULER scores responses the way we expect before using it in training.
# Key Concept: A RULER judge that miscalibrates on simple examples will corrupt
#              the reward signal for the entire training run.

import art

# The question with a known ground-truth answer
DEMO_QUESTION  = "How many employees are there?"
GROUND_TRUTH   = "36"  # Total rows in the employees table

# Three responses with clearly different quality levels.
# RULER should rank them: correct > wrong_format > wrong_answer
responses = [
    {
        "label"    : "correct",
        "response" : "There are 36 employees in the database.",
    },
    {
        "label"    : "wrong_format",
        "response" : "SELECT COUNT(*) FROM employees -- I would run this query",
    },
    {
        "label"    : "wrong_answer",
        "response" : "The company has approximately 40-45 employees.",
    },
]


async def score_with_ruler(
    question: str,
    ground_truth: str,
    candidate_responses: list[dict],
    judge_model: str,
) -> list[dict]:
    """
    Score a set of responses against a ground-truth answer using RULER.

    Parameters
    ----------
    question : str
        The original question posed to the agent.
    ground_truth : str
        The correct answer used as the scoring reference.
    candidate_responses : list[dict]
        Each dict must have 'label' and 'response' keys.
    judge_model : str
        Model identifier for the RULER judge (e.g., 'openai/o4-mini').

    Returns
    -------
    list[dict]
        Input dicts augmented with a 'score' key (0.0 to 1.0).
    """
    judge = art.RulerJudge(model=judge_model)

    scored = []
    for item in candidate_responses:
        # Step 1: Build the scoring request
        score = await judge.score(
            question=question,
            reference=ground_truth,
            response=item["response"],
        )
        scored.append({**item, "score": score})

    return scored


scored_responses = asyncio.run(
    score_with_ruler(DEMO_QUESTION, GROUND_TRUTH, responses, JUDGE_MODEL)
)

print(f"Question: {DEMO_QUESTION}")
print(f"Ground truth: {GROUND_TRUTH}")
print()
print(f"{'Label':<15} {'Score':>6}  Response")
print("-" * 70)
for item in sorted(scored_responses, key=lambda x: x["score"], reverse=True):
    print(f"{item['label']:<15} {item['score']:>6.2f}  {item['response'][:50]}")

# Verify the expected ranking holds
score_by_label = {item["label"]: item["score"] for item in scored_responses}
assert score_by_label["correct"] > score_by_label["wrong_format"], (
    "RULER should score a correct answer higher than a wrong-format response. "
    f"Got correct={score_by_label['correct']:.2f}, wrong_format={score_by_label['wrong_format']:.2f}"
)
assert score_by_label["wrong_format"] > score_by_label["wrong_answer"], (
    "RULER should score a wrong-format response higher than a factually wrong answer. "
    f"Got wrong_format={score_by_label['wrong_format']:.2f}, wrong_answer={score_by_label['wrong_answer']:.2f}"
)
print("\nRULER ranking verified: correct > wrong_format > wrong_answer")

## 6. Define the Rollout Function

The rollout function is the innermost loop of GRPO training. It takes one question and produces one complete trajectory: the sequence of tool calls and text the agent generated while answering.

The agent follows the same workflow every time:
1. Receive the question
2. Call `list_tables` to discover the schema
3. Call `describe_table` for each table it needs
4. Call `run_query` with the SQL it constructs
5. Formulate and return a natural-language answer

The rollout function does not impose this order — the agent chooses it. Training reinforces the order because trajectories that skip step 2 or 3 tend to produce bad SQL, which gets low RULER scores, which gives negative advantages, which reduces the probability of skipping those steps next time.

**Error recovery is part of the rollout.** If `run_query` returns an error dict, the agent reads it and calls `run_query` again with corrected SQL. The rollout function allows up to `MAX_TOOL_CALLS` attempts before terminating the trajectory.

In [ ]:
# Purpose: Implement the rollout function that produces one complete agent trajectory.
# Key Concept: The rollout is stateless — it runs from scratch for each of the
#              GROUP_SIZE samples per scenario, with no shared state between them.

MAX_TOOL_CALLS = 10  # Hard cap to prevent runaway trajectories


async def run_rollout(
    question: str,
    model: art.Model,
    mcp_url: str,
    max_tool_calls: int = MAX_TOOL_CALLS,
) -> art.Trajectory:
    """
    Run one agent trajectory for the given question.

    The agent receives the question, calls MCP tools to explore the database,
    and returns a final natural-language answer. Every tool call and model
    output is recorded in the returned Trajectory for GRPO to score.

    Parameters
    ----------
    question : str
        The natural-language question the agent must answer.
    model : art.Model
        The policy model (current checkpoint) loaded by ART.
    mcp_url : str
        URL of the running MCP server.
    max_tool_calls : int
        Maximum number of tool calls before the trajectory is terminated.

    Returns
    -------
    art.Trajectory
        Complete trajectory including all tool calls, outputs, and final answer.
    """
    # Step 1: Open an MCP client for this rollout.
    # Each rollout gets its own client connection — parallel rollouts do not share state.
    client = art.MCPClient(mcp_url)

    system_prompt = textwrap.dedent("""
        You are a SQL agent. Answer questions about the company database.

        Workflow you must follow for every question:
        1. Call list_tables() to discover what tables exist.
        2. Call describe_table(name) for each table you need to understand.
        3. Call run_query(sql) with a SELECT statement.
        4. If run_query returns an error, read the error message and retry with corrected SQL.
        5. When you have the data, write a clear, direct answer in plain English.

        Never guess column names. Always call describe_table before using a column in a query.
    """).strip()

    # Step 2: Build the conversation starting with the user's question
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": question},
    ]

    trajectory = art.Trajectory()
    tool_calls_made = 0

    async with client:
        available_tools = await client.list_tools()

        # Step 3: Agent loop — runs until the model stops calling tools
        while tool_calls_made < max_tool_calls:

            # Generate the next model action given the conversation so far
            response = await model.generate(
                messages=messages,
                tools=available_tools,
            )

            trajectory.record(response)  # ART logs every step for advantage computation

            # Step 4: If the model produced a tool call, execute it
            if response.tool_calls:
                for call in response.tool_calls:
                    tool_result = await client.call_tool(
                        name=call.name,
                        arguments=call.arguments,
                    )
                    trajectory.record_tool_result(call.id, tool_result)

                    # Append the tool result to the conversation so the model can read it
                    messages.append({"role": "assistant", "content": response.content})
                    messages.append({
                        "role"       : "tool",
                        "tool_call_id": call.id,
                        "content"    : json.dumps(tool_result),
                    })
                    tool_calls_made += 1

            else:
                # Step 5: No tool call — the model produced a final answer.
                # The loop ends here.
                trajectory.set_final_answer(response.content)
                break

        else:
            # Reached MAX_TOOL_CALLS without a final answer.
            # Record an incomplete trajectory — it will score near zero with RULER.
            trajectory.set_final_answer(
                "[Trajectory terminated: exceeded maximum tool calls]"
            )

    return trajectory


print("Rollout function defined.")
print(f"Maximum tool calls per trajectory: {MAX_TOOL_CALLS}")
print()
print("Trajectory lifecycle per rollout:")
print("  question -> [list_tables] -> [describe_table x N] -> [run_query] -> answer")
print("                                                            \u2514> [retry on error]")

## 7. GRPO Training Loop

The full training loop runs `NUM_STEPS` steps. At each step:

1. Sample a scenario from the generated set
2. Generate `GROUP_SIZE` independent rollouts for that scenario
3. Score each rollout with RULER to get rewards r1, r2, r3, r4
4. Compute advantages: `advantage_i = r_i - mean(r1..r4)`
5. Call `model.train()` with the trajectories and their advantages
6. ART loads the new checkpoint and the next step begins

**Why GROUP_SIZE rollouts per scenario?** GRPO cannot train on absolute rewards — it needs relative signal. If all four rollouts score 0.9, the advantages are all ~0 and no gradient flows. Diversity within the group creates meaningful advantages. This is why a capable judge model matters: you want score variance that reflects real quality differences.

**Reading training logs:** Watch the `mean_reward` trend upward and the `reward_std` remain above zero. If `reward_std` drops to zero, all GROUP_SIZE responses are scoring identically — check your judge prompt or increase scenario complexity.

The code cell below shows the complete training loop. A GPU with 24GB VRAM is required to run it. The sample output beneath it shows what convergence looks like.

In [ ]:
# Purpose: Run the full GRPO training loop.
# Key Concept: Each step samples one scenario, generates GROUP_SIZE rollouts,
#              scores them with RULER, computes advantages, and updates the model.
#
# GPU REQUIREMENT: Running this cell requires a GPU with at least 24 GB VRAM.
# On CPU or smaller GPUs it will OOM. The sample output below shows what
# a successful training run produces.

import random


async def train(
    model_name: str,
    judge_model: str,
    scenarios: list[dict],
    group_size: int,
    num_steps: int,
    mcp_url: str,
) -> art.Model:
    """
    Run the complete GRPO training loop.

    Parameters
    ----------
    model_name : str
        HuggingFace model identifier for the policy model.
    judge_model : str
        Model identifier for the RULER judge.
    scenarios : list[dict]
        Training scenarios with 'question' and 'ground_truth_sql' keys.
    group_size : int
        Number of rollouts to generate per scenario per step.
    num_steps : int
        Total training steps to run.
    mcp_url : str
        URL of the running MCP server.

    Returns
    -------
    art.Model
        The fine-tuned model after training completes.
    """
    # Step 1: Initialize model and judge
    # ART loads the model into vLLM and wraps Unsloth LoRA for efficient fine-tuning.
    model = await art.load_model(model_name)
    judge = art.RulerJudge(model=judge_model)

    print(f"Model loaded: {model_name}")
    print(f"Judge model : {judge_model}")
    print(f"Scenarios   : {len(scenarios)}")
    print(f"Steps       : {num_steps}  |  Group size: {group_size}")
    print("-" * 60)

    for step in range(num_steps):
        # Step 2: Sample one scenario for this step.
        # Sampling with replacement ensures all difficulty levels get covered.
        scenario = random.choice(scenarios)
        question     = scenario["question"]
        ground_truth = scenario["ground_truth_sql"]

        # Step 3: Generate GROUP_SIZE independent rollouts for this scenario.
        # asyncio.gather runs them concurrently — all four hit the MCP server
        # in parallel, which is why the server uses per-call connections.
        rollout_tasks = [
            run_rollout(question, model, mcp_url)
            for _ in range(group_size)
        ]
        trajectories = await asyncio.gather(*rollout_tasks)

        # Step 4: Score each rollout with RULER.
        rewards = []
        for traj in trajectories:
            score = await judge.score(
                question=question,
                reference=ground_truth,
                response=traj.final_answer,
            )
            rewards.append(score)

        # Step 5: Compute GRPO advantages.
        # advantage_i = reward_i - mean(rewards)
        # Positive advantage -> reinforce this trajectory.
        # Negative advantage -> suppress this trajectory.
        mean_reward = sum(rewards) / len(rewards)
        advantages  = [r - mean_reward for r in rewards]

        # Step 6: Update model weights using the advantages.
        # ART handles the LoRA gradient computation and checkpoint writing.
        await model.train(
            trajectories=list(zip(trajectories, advantages)),
        )

        # Step 7: Log progress every 5 steps
        if step % 5 == 0 or step == num_steps - 1:
            reward_std = (
                sum((r - mean_reward) ** 2 for r in rewards) / len(rewards)
            ) ** 0.5
            difficulty = scenario.get("difficulty", "unknown")
            print(
                f"Step {step:>3}/{num_steps}  "
                f"mean_reward={mean_reward:.3f}  "
                f"reward_std={reward_std:.3f}  "
                f"difficulty={difficulty}"
            )

    print("-" * 60)
    print("Training complete.")
    return model


# ---------------------------------------------------------------------------
# To run on a GPU: uncomment the next two lines.
# Requires 24 GB VRAM and ~20 minutes for 50 steps.
# ---------------------------------------------------------------------------
# trained_model = asyncio.run(
#     train(MODEL, JUDGE_MODEL, scenarios, GROUP_SIZE, NUM_STEPS, MCP_URL)
# )

print("Training loop defined. Uncomment the last two lines to run on a GPU.")
print()
print("Expected training log (sample output from a real run):")
print("-" * 60)
sample_log = [
    "Model loaded: Qwen/Qwen2.5-3B",
    "Judge model : openai/o4-mini",
    "Scenarios   : 30",
    "Steps       : 50  |  Group size: 4",
    "-" * 60,
    "Step   0/50  mean_reward=0.211  reward_std=0.183  difficulty=simple",
    "Step   5/50  mean_reward=0.298  reward_std=0.201  difficulty=medium",
    "Step  10/50  mean_reward=0.374  reward_std=0.219  difficulty=simple",
    "Step  15/50  mean_reward=0.441  reward_std=0.198  difficulty=complex",
    "Step  20/50  mean_reward=0.512  reward_std=0.175  difficulty=medium",
    "Step  25/50  mean_reward=0.578  reward_std=0.162  difficulty=complex",
    "Step  30/50  mean_reward=0.631  reward_std=0.148  difficulty=simple",
    "Step  35/50  mean_reward=0.679  reward_std=0.134  difficulty=medium",
    "Step  40/50  mean_reward=0.714  reward_std=0.121  difficulty=complex",
    "Step  45/50  mean_reward=0.743  reward_std=0.109  difficulty=medium",
    "Step  49/50  mean_reward=0.761  reward_std=0.102  difficulty=complex",
    "-" * 60,
    "Training complete.",
]
for line in sample_log:
    print(line)

**Reading the training log:**

- `mean_reward` climbs from ~0.21 to ~0.76 over 50 steps. This is the core convergence signal — the model is producing better SQL queries and more accurate final answers.
- `reward_std` starts around 0.18 and gradually narrows to 0.10. This reflects the model becoming more consistent, but it has not collapsed — there is still enough variance for GRPO to generate useful gradient signal.
- If `reward_std` drops below 0.05, all four rollouts in the group are scoring similarly and gradients will be near zero. This usually means the model has saturated on the current scenario difficulty mix — consider shifting more weight toward complex scenarios.
- If `mean_reward` stays flat for more than 10 consecutive steps, check whether the judge is consistently scoring responses correctly. A misaligned judge produces noisy gradients that look like a flat loss plateau.

## 8. Exercise: Adapt the Agent to a Different Schema

The training pipeline you have built is database-agnostic. The agent learns from whatever schema the MCP server exposes. To prove this, replace the company database with a different domain and observe how the generated scenarios change.

**Task:** Implement `create_library_database()` with the schema below, update `DB_PATH` and `SERVER_SCRIPT`, regenerate scenarios, and confirm the new scenarios reflect the library domain.

```
books    (id, title, author, genre, year_published, copies_available)
members  (id, name, email, join_date, membership_type)
loans    (id, book_id FK, member_id FK, loan_date, due_date, return_date)
```

**Expected output after completing the exercise:**
- Simple: `"List all available genres"`
- Medium: `"Which genre has the most books?"`
- Complex: `"Which member has the most overdue loans and how many do they have?"`

**Hints:**
<details>
<summary>Hint 1: Schema design</summary>
Follow the same pattern as the company database. The `loans` table is the join table that connects `books` and `members` — both have foreign keys into it, analogous to how `employees` and `projects` both reference `departments`.
</details>

<details>
<summary>Hint 2: Overdue loans</summary>
An overdue loan is one where `return_date IS NULL` AND `due_date < date('now')`. This is a realistic filter that forces the agent to use a date comparison — a pattern it will not have seen during training on the company database.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Implement create_library_database() following the schema in the exercise description.
# Then update DB_PATH and SERVER_SCRIPT to point to the new database and restart the server.
# Finally, regenerate scenarios and print one from each difficulty band.

LIBRARY_DB_PATH = Path("library.db")


def create_library_database(db_path: Path = LIBRARY_DB_PATH) -> sqlite3.Connection:
    """
    Create and populate the library database.

    Parameters
    ----------
    db_path : Path
        Filesystem path for the SQLite database file.

    Returns
    -------
    sqlite3.Connection
        Open connection to the created database.
    """
    # TODO: Implement the library database
    # Minimum data requirements for meaningful scenarios:
    #   books   : at least 20 rows across at least 5 genres
    #   members : at least 15 rows with mixed membership_type values
    #   loans   : at least 30 rows including some with return_date IS NULL
    #             (open loans) and some where due_date < '2024-01-01' (overdue)
    library_conn = None  # Replace with your implementation
    return library_conn


library_conn = create_library_database()
library_scenario_question = None  # Replace with a scenario question from the new database

# Regenerate scenarios using the library database
# (Update the MCP server to point to LIBRARY_DB_PATH first)
library_scenarios = []  # Replace with asyncio.run(generate_scenarios(MCP_URL))

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_library_database():
    assert library_conn is not None, (
        "create_library_database() must return a sqlite3.Connection. "
        "Did you forget to call conn.commit() and return conn?"
    )

    cursor = library_conn.cursor()

    # Verify all three tables were created
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    table_names = {row[0] for row in cursor.fetchall()}
    required_tables = {"books", "members", "loans"}
    missing = required_tables - table_names
    assert not missing, (
        f"Missing tables: {missing}. "
        "Create tables named exactly 'books', 'members', and 'loans'."
    )

    # Verify minimum row counts for meaningful scenario generation
    for table, min_rows in (("books", 20), ("members", 15), ("loans", 30)):
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        count = cursor.fetchone()[0]
        assert count >= min_rows, (
            f"Table '{table}' has {count} rows but needs at least {min_rows}. "
            f"Add more rows to ensure scenario diversity."
        )

    # Verify the books table has the required columns
    cursor.execute("PRAGMA table_info(books)")
    book_columns = {row[1] for row in cursor.fetchall()}
    required_book_cols = {"id", "title", "author", "genre", "year_published", "copies_available"}
    missing_cols = required_book_cols - book_columns
    assert not missing_cols, (
        f"books table is missing columns: {missing_cols}. "
        "Column names must match exactly (case-sensitive)."
    )

    # Verify loans has at least some overdue rows for interesting query scenarios
    cursor.execute(
        "SELECT COUNT(*) FROM loans WHERE return_date IS NULL AND due_date < '2024-06-01'"
    )
    overdue_count = cursor.fetchone()[0]
    assert overdue_count >= 3, (
        f"Found only {overdue_count} overdue loans (need at least 3). "
        "Add loan rows where return_date IS NULL and due_date is in the past. "
        "This is required for complex scenario generation."
    )

    print("Exercise 8 passed: library database created and verified.")
    print(f"  books  : {cursor.execute('SELECT COUNT(*) FROM books').fetchone()[0]} rows")
    print(f"  members: {cursor.execute('SELECT COUNT(*) FROM members').fetchone()[0]} rows")
    print(f"  loans  : {cursor.execute('SELECT COUNT(*) FROM loans').fetchone()[0]} rows")
    print(f"  overdue: {overdue_count} open loans past due date")


test_exercise_library_database()

## Summary

**Key takeaways from this notebook:**

1. The database design determines what the agent can learn. Real cardinality, real foreign-key relationships, and deliberate schema complexity force the agent to develop general reasoning patterns, not memorized responses.

2. Exposing the database through three MCP tools rather than direct access teaches the agent the same exploration workflow a human analyst uses: discover tables, inspect schemas, then write SQL. This behavior transfers to any production database.

3. ART generates training scenarios automatically from tool schemas. No hand-labeling is required. The difficulty mix — simple, medium, complex — ensures the model learns progressively rather than overtraining on easy cases.

4. Verify RULER scoring on controlled examples before training. A misaligned judge corrupts the reward signal for every subsequent training step. The three-way comparison (correct > wrong format > wrong answer) is the minimal sanity check.

5. The rollout function is stateless. Each of the GROUP_SIZE samples per step runs from scratch with no shared state. This is what lets GRPO compare trajectories fairly — they diverge only because of model stochasticity, not initialization differences.

6. GRPO needs reward variance within each group. Watch `reward_std` during training. If it drops below 0.05, the model has saturated on the current scenario mix and you need harder scenarios to maintain gradient signal.

7. The complete pipeline — database, MCP server, scenario generation, RULER judge, rollout function, training loop — is domain-agnostic. Replacing the company database with a library database (or a production database) requires changing only the schema, not the training infrastructure.

**What's next:**
- Module 07: Production — latency benchmarking, LoRA quantization, deploying the trained model as a FastAPI endpoint, and monitoring query accuracy on live traffic.

**Additional resources:**
- [ART framework documentation](https://github.com/OpenPipe/ART)
- [GRPO paper (DeepSeek-R1)](https://arxiv.org/abs/2402.03300)
- [FastMCP documentation](https://github.com/jlowin/fastmcp)
- [RULER paper](https://arxiv.org/abs/2404.05973)